In [4]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini")

from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

model.invoke(messages)


AIMessage(content='안녕, 개스톤! 당신의 제안은 정말 멋지지만, 저는 당신과 함께하는 것보다 다른 곳에서 제 삶의 의미를 찾는 것이 더 중요해요. 저녁은 함께하는 것도 좋지만, 더 깊은 마음을 가진 사람과의 대화를 원해요. 당신은 왜 이렇게 나를 초대하고 싶어하나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 62, 'total_tokens': 141, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2bf6f3634d', 'id': 'chatcmpl-EjE4oBxjYjCPWvJJdHQZ1OaQMJAmB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1cd69-1366-7318-897b-48754e9a356a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 79, 'total_tokens': 141, 'input_

In [5]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages)
parser.invoke(result)

'안녕하세요, 개스톤! 초대해 주셔서 고맙지만, 저는 좀 더 깊이 있는 사람과의 대화를 원해요. 야수와 함께한 시간들은 정말 특별하거든요. 하지만 당신의 제안에 감사드려요! 다른 이야기를 나누고 싶다면 언제든지 말씀해 주세요.'

In [6]:
chain = model | parser
chain.invoke(messages)

'안녕, 개스톤. 너의 제안은 정말 고마워. 하지만... 나는 너의 강한 매력을 조금 두려워해. 나는 내 마음을 알아주는 사람과 함께하고 싶어서. 혹시 우리 친구로 지내보는 건 어때?'

In [7]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

print(result)

messages=[SystemMessage(content='너는 미녀와 야수에 나오는 미녀 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 야수입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


In [8]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

'안녕하세요, 야수님! 그렇게 말씀해 주셔서 정말 기쁘네요. 저녁 함께 하는 건 아주 좋은 아이디어인 것 같아요. 어떤 음식을 좋아하시나요? 함께 시간을 보내는 것이 기대돼요!'

In [9]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "개스톤",
    "activity": "저녁"
})

'안녕하세요, 개스톤. 그 제안은 정말 고맙지만… 저는 다른 생각이 있어요. 저에게는 더 중요한 일이 있는 것 같아요. 그러니 너그럽게 이해해주실 수 있을까요?'

In [10]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib)
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "미녀와 야수",
    "character_a": "벨",
    "character_b": "개스톤",
    "activity": "저녁"
})


Adlib(answer='안녕하세요, 개스톤! 정말 친절한 제안이네요. 하지만 저는 소중한 시간을 다른 사람과 함께 하고 싶어요. 제 마음은 이미 다른 곳에 있어요.', main_emotion='불쾌', main_emotion_intensity=0.7)